# M6 · Lab 4 — Combined Monitoring Pipeline (exploration)
### Module 6 — Monitoring, Testing & Drift Detection | Spine Project: Truck Delay Classification

| | |
|---|---|
| **Duration** | 60 min |
| **Difficulty** | Intermediate |
| **Tools** | Python 3.12.10, `great-expectations`, `evidently`, `boto3`, pandas |
| **AWS** | **SNS** (publish to the Lab 1 topic — optional; dry-run by default) |
| **Prerequisite** | Lab 1 (SNS topic), Lab 2 (drift), Lab 3 (GE suite). Same `data/` folder. |
| **Builds toward** | **Lab 5** turns this into a production `.py` script; the Airflow branch schedules it; M8 makes it a SageMaker step |
| **Cost** | ₹0 — SNS publishes are free under the monthly free tier |

---
This notebook is where you **compose** the two monitors from Labs 2–3 into one decision and wire it to alerting —
interactively, so you can watch each piece fire. Once the logic is right here, **Lab 5** ships the exact same functions as
a runnable production script. *Notebook to compose → script to operate.*

## 💻 Where to run this — SageMaker (recommended in class), Colab, or local

**In class: the SageMaker Notebook Instance your instructor provisioned** (`m6-truck-delay-monitoring`). Everything is
pre-installed (`evidently`, `great-expectations`, `xgboost`) and the real `data/` folder is already there — just open this
notebook and run.
- **Kernel:** `conda_python3`
- **Data:** `data/reference/final_features.csv` + `data/artifacts/` ship with the labs on the instance (the `M6_DATA_DIR` default is `data`).
- **Auto-stop:** the instance stops itself after ~45 min idle to save cost — your work, installed packages, and saved
  artifacts persist on the EBS volume. Just hit **Start** the next day; **M7 and M8 reuse this same instance.**
- **AWS access (Labs 4/5):** the instance's IAM role already grants `sns:Publish`, so there are **no access keys to configure.**

**Portable fallback (self-paced / post-course):** this notebook also runs on **Google Colab** or **local Jupyter** — the
setup cell below detects the environment and pip-installs what's missing. On Colab, upload the `data/` folder (or set
`M6_DATA_DIR`). Locally, use **Python 3.12.10**. The data + model are the *real* M3 artifacts either way — nothing synthetic.

## Learning Objectives
1. Compose two heterogeneous checks (GE → Evidently) into one decision with a clean **exit-code contract**.
2. Reason about **fail-fast ordering** — why GE runs before Evidently.
3. Build a **structured SNS payload** (a plain dict — no classes) that downstream consumers can parse unchanged.
4. Implement **severity routing** and a **`dry_run`** mode for safe testing.
5. Exercise all three paths: healthy, drift alert, and schema-failure alert.

## Business Context
Labs 1–3 each gave you one piece: an alerting channel, a drift report, a validation suite. None alone tells the on-call
engineer whether the model is OK **right now**. This lab builds the thing they actually need — one routine that, given the
latest batch of production data, answers:
- Is the input structurally valid? *(Great Expectations)*
- Is the distribution still in-spec? *(Evidently)*
- If anything's wrong: **page the right person**, with enough context to triage.

## Prerequisites
- **Real reference frame** + **model artifacts** in `data/` (from Lab 2).
- **GE suite** `truck_delay_features` in `./great_expectations/` (from Lab 3 — run Lab 3 first in this folder).
- **SNS topic** from Lab 1 (optional). Without it, the notebook runs in **dry-run** and prints payloads instead of paging.

In [1]:
# ── Environment + config ──────────────────────────────────────────────────────
import sys, subprocess, os
def _pip(*p): subprocess.run([sys.executable, "-m", "pip", "install", "-q", *p], check=True)
for mod, spec in [("evidently", "evidently>=0.4,<0.5"), ("great_expectations", "great-expectations>=0.18,<1.0"),
                  ("boto3", "boto3>=1.34")]:
    try:
        __import__(mod)
    except ImportError:
        print("Installing", spec); _pip(spec)

import json
import numpy as np
import pandas as pd
from datetime import datetime, timezone

DATA_DIR   = os.environ.get("M6_DATA_DIR", "data")
REF_CSV    = os.path.join(DATA_DIR, "reference", "final_features.csv")
META_JSON  = os.path.join(DATA_DIR, "reference", "feature_metadata.json")
ART_DIR    = os.path.join(DATA_DIR, "artifacts")

# SNS / alert config — env-driven, with safe defaults
TOPIC_ARN  = os.environ.get("TOPIC_ARN", "")              # arn:aws:sns:ap-south-1:<acct>:truck-delay-alerts
AWS_REGION = os.environ.get("AWS_REGION", "ap-south-1")
DRY_RUN    = os.environ.get("DRY_RUN", "true").lower() == "true"   # default: don't actually publish
SUITE_NAME = "truck_delay_features"
SERVICE    = "truck-delay-classifier"

ref = pd.read_csv(REF_CSV)
fmeta = json.load(open(META_JSON))
CONTINUOUS, CATEGORICAL, TARGET = fmeta["continuous_features"], fmeta["categorical_features"], fmeta["target"]
print(f"Reference {ref.shape} | dry_run={DRY_RUN} | topic set: {bool(TOPIC_ARN)}")

Reference (12308, 37) | dry_run=True | topic set: False


## Step 1 · The pipeline contract
```
INPUT   : a production-batch DataFrame (same schema as reference) + the reference + the GE suite
PROCESS : 1. GE validate.  fail -> publish(alert_type=ge_validation, severity=critical), return 1
          2. Evidently.    drift -> publish(alert_type=drift, severity=info|warning|critical), return 1
          3. both pass -> log success, return 0
OUTPUT  : exit-code-style int (0 healthy, 1 alert) + a structured dict; SNS publish as a side effect on failure
SAFETY  : dry_run -> print the payload instead of publishing
```
The exit-code contract is what lets cron / Airflow / EventBridge / SageMaker Pipelines consume this unchanged.

## Step 2 · GE validation (cheap → runs first, fails fast)

In [2]:
import great_expectations as gx
from great_expectations.core.batch import RuntimeBatchRequest

# Auto-detect the GE project folder: newer great-expectations 0.18 builds default to
# "gx/", older ones to "great_expectations/". This points Lab 4 at whichever Lab 3 created,
# so it can't break again on the folder name. Override with GE_PROJECT_DIR if needed.
_GE_ROOT = os.environ.get("GE_PROJECT_DIR") or next(
    (p for p in ("./gx", "./great_expectations") if os.path.isdir(p)), "./gx")
_ge_context = gx.get_context(context_root_dir=_GE_ROOT)
print("GE project root:", _ge_context.root_directory)

def run_ge_validation(df: pd.DataFrame, suite_name: str = SUITE_NAME) -> tuple:
    '''Validate df against the Lab 3 suite. Returns (success: bool, details: dict).'''
    if "truck_delay_runtime" not in [d["name"] for d in _ge_context.list_datasources()]:
        _ge_context.add_datasource(
            name="truck_delay_runtime", class_name="Datasource",
            execution_engine={"class_name": "PandasExecutionEngine"},
            data_connectors={"default_runtime_data_connector_name": {
                "class_name": "RuntimeDataConnector", "batch_identifiers": ["default_identifier_name"]}})
    br = RuntimeBatchRequest(
        datasource_name="truck_delay_runtime",
        data_connector_name="default_runtime_data_connector_name",
        data_asset_name="production_batch",
        runtime_parameters={"batch_data": df},
        batch_identifiers={"default_identifier_name": datetime.utcnow().isoformat()})
    res = _ge_context.get_validator(batch_request=br, expectation_suite_name=suite_name).validate()
    details = {
        "evaluated": res["statistics"]["evaluated_expectations"],
        "passed": res["statistics"]["successful_expectations"],
        "failed": res["statistics"]["unsuccessful_expectations"],
        "failures": [{"expectation": r["expectation_config"]["expectation_type"],
                      "column": r["expectation_config"]["kwargs"].get("column"),
                      "unexpected_count": r["result"].get("unexpected_count", 0),
                      "sample": r["result"].get("partial_unexpected_list", [])[:3]}
                     for r in res["results"] if not r["success"]],
    }
    return bool(res["success"]), details

print("GE validator ready (suite:", SUITE_NAME, ")")

GE project root: /home/sagemaker-user/MLOpsself/Module 6 complete/labs/gx
GE validator ready (suite: truck_delay_features )


## Step 3 · Evidently drift (more expensive → runs second)

In [3]:
from evidently import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, TargetDriftPreset

# Lab 2 used Evidently's default 50%-of-columns threshold for the dataset-drift flag
# (exploration). For the production ALERT path we lower it to 10% so a realistic shift —
# the monsoon batch drifts ~20% of columns — actually trips a dataset-level alert.
DRIFT_SHARE_THRESHOLD = float(os.environ.get("DRIFT_SHARE_THRESHOLD", "0.1"))

def run_drift_detection(reference: pd.DataFrame, current: pd.DataFrame) -> tuple:
    '''Returns (no_drift: bool, details: dict). no_drift=True means everything is fine.'''
    cm = ColumnMapping(target=TARGET, prediction=None,
                       numerical_features=CONTINUOUS, categorical_features=CATEGORICAL)
    rep = Report(metrics=[DataDriftPreset(drift_share=DRIFT_SHARE_THRESHOLD), TargetDriftPreset()])
    rep.run(reference_data=reference, current_data=current, column_mapping=cm)
    md = rep.as_dict()["metrics"]

    # metrics[0] = DatasetDriftMetric summary. Its "drift_share" is the THRESHOLD; the actual
    # fraction drifted is "share_of_drifted_columns". Per-column table is on DataDriftTable.
    summary = md[0]["result"]
    table = next((m["result"]["drift_by_columns"]
                  for m in md if "drift_by_columns" in m.get("result", {})), {})
    details = {
        "dataset_drift": bool(summary["dataset_drift"]),
        "drift_share": float(summary["share_of_drifted_columns"]),
        "n_drifted": summary["number_of_drifted_columns"],
        "n_total": summary["number_of_columns"],
        "drifted_columns": [c for c, cr in table.items() if cr["drift_detected"]][:10],
    }
    return (not details["dataset_drift"]), details

print("Drift detector ready.")


Drift detector ready.


## Step 4 · Severity routing + alert payload (plain dict — no classes)

In [4]:
def route_severity(alert_type: str, drift_share: float = 0.0) -> str:
    '''GE failures are always critical; drift severity scales with how much drifted.'''
    if alert_type == "ge_validation":
        return "critical"
    if drift_share > 0.50:
        return "critical"
    if drift_share > 0.30:
        return "warning"
    return "info"

def build_alert_payload(alert_type: str, severity: str, summary: str, details: dict) -> dict:
    '''The structured message every consumer (email, Slack-via-Lambda, M8 trigger) parses. A dict, not a class.'''
    return {
        "schema_version": "1.0",
        "alert_type": alert_type,           # ge_validation | drift
        "severity": severity,               # info | warning | critical
        "service": SERVICE,
        "environment": os.environ.get("ENVIRONMENT", "production"),
        "detected_at": datetime.now(timezone.utc).isoformat(),
        "summary": summary,
        "details": details,
        "runbook_url": f"https://wiki.freshbasket.in/runbooks/{alert_type}",
    }

## Step 5 · SNS publish (dry-run aware)

In [5]:
def publish_alert(payload: dict, dry_run: bool = True):
    '''Publish to SNS, or print the payload when dry_run. Returns the MessageId or None.'''
    subject = f"[{payload['severity'].upper()}] {payload['service']} {payload['alert_type']}"
    body = json.dumps(payload, indent=2)
    if dry_run or not TOPIC_ARN:
        print("── DRY RUN — would publish to", TOPIC_ARN or "<no TOPIC_ARN set>")
        print("Subject:", subject); print(body)
        return None
    import boto3
    resp = boto3.client("sns", region_name=AWS_REGION).publish(
        TopicArn=TOPIC_ARN, Subject=subject[:99], Message=body,
        MessageAttributes={k: {"DataType": "String", "StringValue": payload[k]}
                           for k in ("alert_type", "severity", "service")})
    return resp["MessageId"]

## Step 6 · Orchestrate — the combined check (GE → Evidently → alert)

In [6]:
def run_monitoring(current: pd.DataFrame, reference: pd.DataFrame = ref, dry_run: bool = True) -> int:
    '''Full monitor for one batch. Returns 0 (healthy) or 1 (alert published). Mirrors Lab 5's main().'''
    print(f"[start] {datetime.now(timezone.utc).isoformat()}  batch={current.shape}")

    # 1) GE first — cheap, fail fast on schema
    ge_ok, ge = run_ge_validation(current)
    print(f"[ge]    success={ge_ok}  passed={ge['passed']}/{ge['evaluated']}")
    if not ge_ok:
        payload = build_alert_payload(
            "ge_validation", route_severity("ge_validation"),
            f"Great Expectations failed: {ge['failed']}/{ge['evaluated']} expectations broke",
            {"ge": ge})
        publish_alert(payload, dry_run)
        return 1

    # 2) Evidently drift — only if the data is structurally sound
    no_drift, dr = run_drift_detection(reference, current)
    print(f"[drift] dataset_drift={not no_drift}  drift_share={dr['drift_share']:.2%}")
    if not no_drift:
        payload = build_alert_payload(
            "drift", route_severity("drift", dr["drift_share"]),
            f"Data drift: {dr['n_drifted']}/{dr['n_total']} columns ({dr['drift_share']:.0%} of features)",
            {"drift": dr})
        publish_alert(payload, dry_run)
        return 1

    print("[done]  all checks passed — no alert")
    return 0

## Step 7 · Build production batches with the Lab 2 / Lab 3 simulators
We reuse the *same* drift + corruption simulators from Labs 2–3 (the only synthetic part — there's no live traffic in
class). In production, `current` would be the last hour of inferences read from CloudWatch Logs / S3.

In [7]:
def make_drifted_batch(reference, n=2000, seed=42):
    '''Monsoon drift (same as Lab 2): precip/humidity up, fleet aging, heavy-rain, more delays.'''
    rng = np.random.default_rng(seed)
    cur = reference.sample(n=n, random_state=seed).reset_index(drop=True).copy()
    for p in ("route", "origin", "dest"):
        cur[f"{p}_avg_precip"] = cur[f"{p}_avg_precip"] * rng.uniform(1.3, 1.8, n)
        cur[f"{p}_avg_humidity"] = (cur[f"{p}_avg_humidity"] + rng.normal(8, 2, n)).clip(0, 100)
    cur["truck_age"] = (cur["truck_age"] + rng.integers(0, 4, n)).clip(1, 30)
    heavy = rng.random(n) < 0.30
    cur.loc[heavy, "route_description"] = rng.choice(["Heavy rain", "Moderate rain", "Patchy rain possible"], heavy.sum())
    flip = (cur[TARGET] == 0) & (rng.random(n) < 0.18); cur.loc[flip, TARGET] = 1
    return cur

def make_corrupt_batch(reference, n=500, seed=7):
    '''Schema corruption (same as Lab 3): negative age, null ratings, unknown fuel, bad target.'''
    b = reference.sample(n=n, random_state=seed).reset_index(drop=True).copy()
    b.loc[b.index[:25], "truck_age"] = -3
    b.loc[b.sample(frac=0.05, random_state=1).index, "ratings"] = np.nan
    b.loc[b.index[300:310], "fuel_type"] = "hydrogen"
    b.loc[b.index[400:402], TARGET] = 2
    return b
print("Simulators ready.")

Simulators ready.


## Step 8 · Exercise all three paths

In [8]:
print("════════ PATH 1 — DRIFT (synthetic monsoon batch) ════════")
rc1 = run_monitoring(make_drifted_batch(ref), dry_run=True)
print("exit code:", rc1, "\n")

════════ PATH 1 — DRIFT (synthetic monsoon batch) ════════
[start] 2026-06-25T09:22:32.455834+00:00  batch=(2000, 37)


Calculating Metrics:   0%|          | 0/452 [00:00<?, ?it/s]

[ge]    success=False  passed=166/278
── DRY RUN — would publish to <no TOPIC_ARN set>
Subject: [CRITICAL] truck-delay-classifier ge_validation
{
  "schema_version": "1.0",
  "alert_type": "ge_validation",
  "severity": "critical",
  "service": "truck-delay-classifier",
  "environment": "production",
  "detected_at": "2026-06-25T09:22:38.787447+00:00",
  "summary": "Great Expectations failed: 112/278 expectations broke",
  "details": {
    "ge": {
      "evaluated": 278,
      "passed": 166,
      "failed": 112,
      "failures": [
        {
          "expectation": "expect_table_row_count_to_be_between",
          "column": null,
          "unexpected_count": 0,
          "sample": []
        },
        {
          "expectation": "expect_column_mean_to_be_between",
          "column": "route_avg_temp",
          "unexpected_count": 0,
          "sample": []
        },
        {
          "expectation": "expect_column_proportion_of_unique_values_to_be_between",
          "column": "rou

In [9]:
print("════════ PATH 2 — SCHEMA FAILURE (corrupted batch, GE fails first) ════════")
rc2 = run_monitoring(make_corrupt_batch(ref), dry_run=True)
print("exit code:", rc2, "\n")

════════ PATH 2 — SCHEMA FAILURE (corrupted batch, GE fails first) ════════
[start] 2026-06-25T09:22:47.230983+00:00  batch=(500, 37)


Calculating Metrics:   0%|          | 0/452 [00:00<?, ?it/s]

[ge]    success=False  passed=152/278
── DRY RUN — would publish to <no TOPIC_ARN set>
Subject: [CRITICAL] truck-delay-classifier ge_validation
{
  "schema_version": "1.0",
  "alert_type": "ge_validation",
  "severity": "critical",
  "service": "truck-delay-classifier",
  "environment": "production",
  "detected_at": "2026-06-25T09:22:53.369002+00:00",
  "summary": "Great Expectations failed: 126/278 expectations broke",
  "details": {
    "ge": {
      "evaluated": 278,
      "passed": 152,
      "failed": 126,
      "failures": [
        {
          "expectation": "expect_table_row_count_to_be_between",
          "column": null,
          "unexpected_count": 0,
          "sample": []
        },
        {
          "expectation": "expect_column_min_to_be_between",
          "column": "route_avg_temp",
          "unexpected_count": 0,
          "sample": []
        },
        {
          "expectation": "expect_column_max_to_be_between",
          "column": "route_avg_temp",
          "

In [10]:
print("════════ PATH 3 — HEALTHY (reference itself: in-distribution + schema-valid) ════════")
rc3 = run_monitoring(ref.sample(n=2000, random_state=99).reset_index(drop=True), dry_run=True)
print("exit code:", rc3)

════════ PATH 3 — HEALTHY (reference itself: in-distribution + schema-valid) ════════
[start] 2026-06-25T09:22:59.907529+00:00  batch=(2000, 37)


Calculating Metrics:   0%|          | 0/452 [00:00<?, ?it/s]

[ge]    success=False  passed=175/278
── DRY RUN — would publish to <no TOPIC_ARN set>
Subject: [CRITICAL] truck-delay-classifier ge_validation
{
  "schema_version": "1.0",
  "alert_type": "ge_validation",
  "severity": "critical",
  "service": "truck-delay-classifier",
  "environment": "production",
  "detected_at": "2026-06-25T09:23:06.066149+00:00",
  "summary": "Great Expectations failed: 103/278 expectations broke",
  "details": {
    "ge": {
      "evaluated": 278,
      "passed": 175,
      "failed": 103,
      "failures": [
        {
          "expectation": "expect_table_row_count_to_be_between",
          "column": null,
          "unexpected_count": 0,
          "sample": []
        },
        {
          "expectation": "expect_column_min_to_be_between",
          "column": "route_avg_temp",
          "unexpected_count": 0,
          "sample": []
        },
        {
          "expectation": "expect_column_max_to_be_between",
          "column": "route_avg_temp",
          "

> **Why GE runs first:** GE is ~50 ms; Evidently is ~3–5 s. And if the data is structurally broken, the Evidently report
> on it is meaningless. Fail fast on the cheap check. Notice PATH 2 never reached the drift stage.

## Step 9 · (Optional) publish for real — verify the Lab 1 email lands

In [11]:
# Set TOPIC_ARN (from Lab 1) + DRY_RUN=false in the config cell, then:
#   run_monitoring(make_drifted_batch(ref), dry_run=False)
# Within ~30 s you should get an email titled "[INFO] truck-delay-classifier drift".
# Kept dry by default so the lab never pages anyone unexpectedly.
print("To publish for real: set TOPIC_ARN + DRY_RUN=false, then run_monitoring(..., dry_run=False)")

To publish for real: set TOPIC_ARN + DRY_RUN=false, then run_monitoring(..., dry_run=False)


## Bridge to Lab 5 — production script
Every function above (`run_ge_validation`, `run_drift_detection`, `route_severity`, `build_alert_payload`,
`publish_alert`, `run_monitoring`) is already written in **functional, no-class** style — so Lab 5 lifts them almost
verbatim into `monitoring_pipeline/` with an `argparse` CLI, an exit-code `main()`, and `--current` / `--simulate` /
`--dry-run` flags. The notebook is for *seeing* each piece; the script is for *running* it on a schedule.

## Verification Checklist
- [ ] GE suite loaded from `./great_expectations/` (Lab 3 output)
- [ ] PATH 1 (drift) returns exit 1 and prints a `drift` payload
- [ ] PATH 2 (corruption) returns exit 1 with a `ge_validation` payload — and never runs the drift stage
- [ ] PATH 3 (healthy) returns exit 0, no payload
- [ ] You can explain fail-fast ordering and write the 8-field payload schema from memory
- [ ] (optional) a real SNS publish delivered an email

## What's next
- **Lab 5** — ship this as a production `.py` package.
- **Branch project (Airflow)** — the same monitor running on a schedule for a Banking-Churn model under Airflow + Docker.
- **M8 capstone** — this monitor becomes a SageMaker Processing step in the automated pipeline.

## Troubleshooting
| Symptom | Fix |
|---|---|
| `get_context` can't find `./great_expectations` | run **Lab 3** in this folder first (it creates the project + suite) |
| `ImportError: ColumnMapping` | pin `evidently>=0.4,<0.5` |
| publish raises `AccessDenied` | the IAM identity needs `sns:Publish` on the topic (Lab 1) |
| email never arrives though exit=1 | the SNS subscription isn't confirmed — check `aws sns list-subscriptions-by-topic` |
| drift on the reference itself | you passed a modified frame as `reference` — reload `final_features.csv` |